In [ ]:
!pip install pyspark

In [ ]:
import os
import tarfile
import urllib.request
import numpy as np
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.types import ArrayType, FloatType, StructType, StructField, IntegerType
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.functions import udf, col

spark = SparkSession.builder \
    .appName("SIFT_DataLoader") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

list_to_vector_udf = udf(lambda l: Vectors.dense(l), VectorUDT())

def read_fvecs(filename):
    vecs = np.fromfile(filename, dtype=np.float32)
    if vecs.size == 0: return np.zeros((0, 0))
    d = vecs.view(np.int32)[0]
    vecs = vecs.reshape(-1, d + 1)
    return vecs[:, 1:]

def load_sift_data(dataset_size="small"):
    dataset_size = dataset_size.lower()
    parquet_base = f"sift_{dataset_size}_base.parquet"
    parquet_query = f"sift_{dataset_size}_query.parquet"

    if os.path.exists(parquet_base) and os.path.exists(parquet_query):
        df_base = spark.read.parquet(parquet_base)
        df_query = spark.read.parquet(parquet_query)
        return df_base, df_query

    os.makedirs("raw_data", exist_ok=True)

    base_fvecs_path = f"raw_data/{dataset_size}_base.fvecs"
    query_fvecs_path = f"raw_data/{dataset_size}_query.fvecs"
    partitions = 100 if dataset_size == "1m" else 4

    if dataset_size == "small":
        if not (os.path.exists(base_fvecs_path) and os.path.exists(query_fvecs_path)):
            url = "https://github.com/TileDB-Inc/TileDB-Vector-Search/releases/download/0.0.1/siftsmall.tgz"
            urllib.request.urlretrieve(url, "raw_data/siftsmall.tgz")

            with tarfile.open("raw_data/siftsmall.tgz", "r:gz") as tar:
                if hasattr(tarfile, 'data_filter'):
                    tar.extractall(path="raw_data/", filter='data')
                else:
                    tar.extractall(path="raw_data/")

            for root, dirs, files in os.walk("raw_data"):
                for file in files:
                    if file == "siftsmall_base.fvecs":
                        shutil.move(os.path.join(root, file), base_fvecs_path)
                    elif file == "siftsmall_query.fvecs":
                        shutil.move(os.path.join(root, file), query_fvecs_path)

    elif dataset_size == "1m":
        if not os.path.exists(base_fvecs_path):
            urllib.request.urlretrieve("https://huggingface.co/datasets/qbo-odp/sift1m/resolve/main/sift_base.fvecs", base_fvecs_path)
        if not os.path.exists(query_fvecs_path):
            urllib.request.urlretrieve("https://huggingface.co/datasets/qbo-odp/sift1m/resolve/main/sift_query.fvecs", query_fvecs_path)

    base_vectors_np = read_fvecs(base_fvecs_path)
    query_vectors_np = read_fvecs(query_fvecs_path)

    schema = StructType([StructField("r_id", IntegerType(), False), StructField("features", ArrayType(FloatType()), False)])
    schema_q = StructType([StructField("s_id", IntegerType(), False), StructField("features", ArrayType(FloatType()), False)])

    base_data = [(i, vec.tolist()) for i, vec in enumerate(base_vectors_np)]
    df_base = spark.createDataFrame(spark.sparkContext.parallelize(base_data, partitions), schema) \
                   .withColumn("features_vec", list_to_vector_udf(col("features"))).drop("features")

    query_data = [(i, vec.tolist()) for i, vec in enumerate(query_vectors_np)]
    df_query = spark.createDataFrame(spark.sparkContext.parallelize(query_data, 4), schema_q) \
                    .withColumn("features_vec", list_to_vector_udf(col("features"))).drop("features")

    df_base.write.mode("overwrite").parquet(parquet_base)
    df_query.write.mode("overwrite").parquet(parquet_query)

    return df_base, df_query

print("Φορτώθηκαν επιτυχώς!")

Φορτώθηκαν επιτυχώς!


In [ ]:
user_input = input("Παρακαλώ εισάγετε την παράμετρο απόστασης θ: ")
theta = float(user_input)
print(f"Το όριο απόστασης ορίστηκε στο: {theta}")

Παρακαλώ εισάγετε την παράμετρο απόστασης θ: 250
Το όριο απόστασης ορίστηκε στο: 250.0


In [ ]:
import time
import numpy as np
from pyspark.sql.functions import col, udf
from pyspark.sql.types import FloatType

df_R, df_S = load_sift_data("small")

df_R_truth = df_R.withColumnRenamed("r_id", "base_id").withColumnRenamed("features_vec", "base_vec")
df_S_truth = df_S.withColumnRenamed("s_id", "query_id").withColumnRenamed("features_vec", "query_vec")

print("Υπολογισμός Ground Truth (Cross Join)")
start_time = time.time()

crossed_df = df_R_truth.crossJoin(df_S_truth)

@udf(returnType=FloatType())
def dist_udf(v1, v2):
    return float(np.linalg.norm(v1.toArray() - v2.toArray()))

ground_truth_results = crossed_df.withColumn("euclidean_dist", dist_udf(col("base_vec"), col("query_vec"))) \
                                 .filter(col("euclidean_dist") <= theta) \
                                 .select("base_id", "query_id", "euclidean_dist")

total_matches = ground_truth_results.count()
end_time = time.time()

print(f"Ολοκληρώθηκε σε: {end_time - start_time:.2f} δευτερόλεπτα")
print(f"Για θ={theta}, βρέθηκαν {total_matches} ζευγάρια.")

ground_truth_results.show(5)

ground_truth_results.write.mode("overwrite").parquet("siftsmall_ground_truth.parquet")
print("Το Ground Truth αποθηκεύτηκε σε Parquet!")

Υπολογισμός Ground Truth (Cross Join)
Ολοκληρώθηκε σε: 70.01 δευτερόλεπτα
Για θ=250.0, βρέθηκαν 1613 ζευγάρια.
+-------+--------+--------------+
|base_id|query_id|euclidean_dist|
+-------+--------+--------------+
|     31|      22|     231.64629|
|     61|      11|     241.64644|
|    105|      22|     208.31706|
|    107|      19|     236.90082|
|    117|      22|       249.948|
+-------+--------+--------------+
only showing top 5 rows
Το Ground Truth αποθηκεύτηκε σε Parquet!


In [ ]:
import time
import pandas as pd
from pyspark.ml.feature import BucketedRandomProjectionLSH

df_R, df_S = load_sift_data("small")
df_truth = spark.read.parquet("siftsmall_ground_truth.parquet")
total_truth = df_truth.count()

df_R.cache()
df_S.cache()
df_truth.cache()

bucket_lengths = [100.0, 200.0, 300.0, 400.0, 500.0]
hash_tables = [1, 2, 3]

results = []

print(f"Grid Search (15 Συνδυασμοί). Συνολικά πραγματικά ζευγάρια: {total_truth}")

for ht in hash_tables:
    for bl in bucket_lengths:

        brp = BucketedRandomProjectionLSH(
            inputCol="features_vec",
            outputCol="hashes",
            bucketLength=bl,
            numHashTables=ht
        )

        start_time = time.time()

        model = brp.fit(df_R)
        approx_matches_df = model.approxSimilarityJoin(df_R, df_S, theta, distCol="euclidean_dist")

        lsh_results = approx_matches_df.select(
            approx_matches_df.datasetA.r_id.alias("base_id"),
            approx_matches_df.datasetB.s_id.alias("query_id")
        )

        total_lsh = lsh_results.count()
        end_time = time.time()
        time_taken = end_time - start_time

        correct_matches = lsh_results.join(df_truth, ["base_id", "query_id"]).count()

        recall = (correct_matches / total_truth) * 100

        print(f"Ολοκληρώθηκε σε {time_taken:.2f}s (Recall: {recall:.2f}%)")

        results.append({
            "Hash Tables": ht,
            "Bucket Length": bl,
            "Χρόνος (sec)": round(time_taken, 2),
            "Βρέθηκαν (Συνολικά)": total_lsh,
            "Σωστά Ζευγάρια": correct_matches,
            "Recall (%)": round(recall, 2)
        })

print("Το Grid Search ολοκληρώθηκε!")

results_df = pd.DataFrame(results)

display(results_df)

Grid Search (15 Συνδυασμοί). Συνολικά πραγματικά ζευγάρια: 1613
Ολοκληρώθηκε σε 3.98s (Recall: 96.09%)
Ολοκληρώθηκε σε 1.43s (Recall: 96.09%)
Ολοκληρώθηκε σε 1.01s (Recall: 96.09%)
Ολοκληρώθηκε σε 1.03s (Recall: 96.09%)
Ολοκληρώθηκε σε 0.96s (Recall: 96.09%)
Ολοκληρώθηκε σε 1.05s (Recall: 99.57%)
Ολοκληρώθηκε σε 0.97s (Recall: 99.57%)
Ολοκληρώθηκε σε 0.95s (Recall: 99.57%)
Ολοκληρώθηκε σε 0.94s (Recall: 99.57%)
Ολοκληρώθηκε σε 1.16s (Recall: 99.57%)
Ολοκληρώθηκε σε 1.19s (Recall: 99.81%)
Ολοκληρώθηκε σε 1.34s (Recall: 99.81%)
Ολοκληρώθηκε σε 1.33s (Recall: 99.81%)
Ολοκληρώθηκε σε 1.19s (Recall: 99.81%)
Ολοκληρώθηκε σε 1.29s (Recall: 99.81%)
Το Grid Search ολοκληρώθηκε!


,Hash Tables,Bucket Length,Χρόνος (sec),Βρέθηκαν (Συνολικά),Σωστά Ζευγάρια,Recall (%)
0,1,100.0,3.98,1550,1550,96.09
1,1,200.0,1.43,1550,1550,96.09
2,1,300.0,1.01,1550,1550,96.09
3,1,400.0,1.03,1550,1550,96.09
4,1,500.0,0.96,1550,1550,96.09
5,2,100.0,1.05,1606,1606,99.57
6,2,200.0,0.97,1606,1606,99.57
7,2,300.0,0.95,1606,1606,99.57
8,2,400.0,0.94,1606,1606,99.57
9,2,500.0,1.16,1606,1606,99.57


In [ ]:
def find_best_combination(df):
    best_row = df.sort_values(by=['Recall (%)', 'Χρόνος (sec)'], ascending=[False, True]).iloc[0]

    print("ΝΙΚΗΤΗΣ (Βέλτιστο Trade-off):")
    print(f"🔹 Hash Tables:   {int(best_row['Hash Tables'])}")
    print(f"🔹 Bucket Length: {best_row['Bucket Length']}")
    print(f"🔹 Recall:        {best_row['Recall (%)']}%")
    print(f"🔹 Χρόνος:        {best_row['Χρόνος (sec)']} sec")
    print("Αυτές είναι οι παράμετροι που πρέπει να χρησιμοποιήσεις στο (SIFT1M)!")

    return best_row

best_params = find_best_combination(results_df)

ΝΙΚΗΤΗΣ (Βέλτιστο Trade-off):
🔹 Hash Tables:   3
🔹 Bucket Length: 100.0
🔹 Recall:        99.81%
🔹 Χρόνος:        1.19 sec
Αυτές είναι οι παράμετροι που πρέπει να χρησιμοποιήσεις στο (SIFT1M)!


In [ ]:
import time
from pyspark.ml.feature import BucketedRandomProjectionLSH

df_R_big, df_S_big_full = load_sift_data("1m")

df_S_big = df_S_big_full.limit(100)

df_R_big.cache()
df_S_big.cache()

print(f"Εκτέλεση. Βάση: {df_R_big.count()} | Queries: {df_S_big.count()}")

brp_fast = BucketedRandomProjectionLSH(inputCol="features_vec", outputCol="hashes", numHashTables=1, bucketLength=300.0)
model_fast = brp_fast.fit(df_R_big)

start_time_1 = time.time()
matches_fast = model_fast.approxSimilarityJoin(df_R_big, df_S_big, theta, distCol="euclidean_dist")
total_fast = matches_fast.count()
end_time_1 = time.time()
time_fast = end_time_1 - start_time_1

brp_acc = BucketedRandomProjectionLSH(inputCol="features_vec", outputCol="hashes", numHashTables=3, bucketLength=400.0)
model_acc = brp_acc.fit(df_R_big)

start_time_2 = time.time()
matches_acc = model_acc.approxSimilarityJoin(df_R_big, df_S_big, theta, distCol="euclidean_dist")
total_acc = matches_acc.count()
end_time_2 = time.time()
time_acc = end_time_2 - start_time_2

print(f"\nΤΕΛΙΚΗ ΣΥΓΚΡΙΣΗ (Στο SIFT1M για θ={theta} και 100 queries)")
print(f"Μοντέλο 1 (Ταχύτητα): Βρήκε {total_fast} ζευγάρια σε {time_fast:.2f} sec")
print(f"Μοντέλο 2 (Ακρίβεια): Βρήκε {total_acc} ζευγάρια σε {time_acc:.2f} sec")

Εκτέλεση. Βάση: 1000000 | Queries: 100

ΤΕΛΙΚΗ ΣΥΓΚΡΙΣΗ (Στο SIFT1M για θ=250.0 και 100 queries)
Μοντέλο 1 (Ταχύτητα): Βρήκε 125416 ζευγάρια σε 26.08 sec
Μοντέλο 2 (Ακρίβεια): Βρήκε 128443 ζευγάρια σε 59.66 sec
